In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [5]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

import smtplib
import secrets
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import re


# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Intelligent Freight Quote Generation", page_icon="🚢", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#F4F8FB",
    "bg_sidebar": "#EAF4FA",
    "bg_card": "#FFFFFF",
    "bg_card_alt": "#DCEEF9",

    "text_main": "#1F2937",
    "text_heading": "#0F172A",
    "text_muted": "#64748B",

    "accent": "#0B5ED7",
    "accent_hover": "#084298",
    "accent_text": "#FFFFFF",

    "border": "#1E3A5F",
    "border_light": "#C7DDEB",

    "success": "#16A34A",
    "danger": "#DC2626"
}

JWT_SECRET =  os.getenv("JWT_SECRET")

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    @keyframes oceanFlow {{
    0% {{ background-position: 0% 50%; }}
    50% {{ background-position: 100% 50%; }}
    100% {{ background-position: 0% 50%; }}
    }}
    html, body, .stApp {{
    background: linear-gradient(
        -45deg,
        #F4F8FB,
        #DCEEF9,
        #EAF4FA,
        #CDE9F7,
        #F4F8FB
    ) !important;

    background-size: 400% 400% !important;
    animation: oceanFlow 15s ease infinite !important;

    font-family: 'Inter', sans-serif !important;
    color: {COLORS['text_main']} !important;
    }}
    /* 🚀 DO NOT HIDE HEADER! Only hide footer and bottom decoration so toggle arrow stays visible! */
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* 🚀 Highlight Streamlit's Native Sidebar Re-Open Button (Top-Left) as a Yellow Badge! */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* Inputs */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

def valid_email(email):
    pattern = r'^[A-Za-z0-9._%+-]{2,}@[A-Za-z0-9.-]{2,}\.[A-Za-z]{2,}$'
    return re.match(pattern, email)

def valid_password(password):
    pattern = r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&^#])[A-Za-z\d@$!%*?&^#]{8,}$'
    return re.match(pattern, password)

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt(os.getenv("ADMIN_PASSWORD")), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

# ==========================
# OTP Helper Functions
# ==========================

def generate_otp():
    return str(secrets.randbelow(900000) + 100000)

def send_otp_email(receiver_email, otp):
    sender_email = os.getenv("EMAIL_ADDRESS")
    sender_password = os.getenv("EMAIL_PASSWORD")

    subject = "Intelligent Freight Quote Generation - Password Reset OTP"

    body = f"""
Hello,

Your OTP for password reset is:

{otp}

This OTP is valid for 5 minutes.

Regards,
Intelligent Freight Quote Generation Team
"""

    msg = MIMEMultipart()
    msg["From"] = sender_email
    msg["To"] = receiver_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    server = smtplib.SMTP("smtp.gmail.com", 587)
    server.starttls()
    server.login(sender_email, sender_password)
    server.send_message(msg)
    server.quit()

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="AI-Powered Maritime Logistics Platform"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Intelligent Freight Quote Generation</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Access Your Freight Management Portal")
            email = st.text_input("Email address", placeholder="Enter your registered email").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
              if email == "" or pwd == "":
                st.error("⚠️ Please fill all fields.")
              else:
                with get_db() as c:
                  r = c.execute(
                      "SELECT password_hash FROM users WHERE email=?",
                       (email,)
                       ).fetchone()
                if r and check_txt(pwd, r[0]):
                  st.session_state.token = make_jwt(email)
                  navigate("Dashboard")
                else:
                  st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Intelligent Freight Quote Generation today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
              if not uname or not email or not pwd or not confirm_pwd or not sa:
                st.error("⚠️ Please fill all mandatory fields.")
              elif not valid_email(email):
                st.error("❌ Enter a valid email (example: ab@cd.ef)")
              elif not valid_password(pwd):
                st.error("""
                Password must contain:
                • Minimum 8 characters
                • One uppercase letter
                • One lowercase letter
                • One number
                • One special character
                """)
              elif pwd != confirm_pwd:
                st.error("❌ Passwords do not match.")
              else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        st.session_state.reset_email = email
                        st.session_state.sq_p = r[0]
                        st.session_state.reset_mode = "sq"
                        st.rerun()
                    else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                  if email.strip() == "":
                    st.error("❌ Please enter your registered email.")
                  else:
                      with get_db() as c:
                        user = c.execute(
                            "SELECT email FROM users WHERE email=?",
                             (email,)
                             ).fetchone()
                      if user:
                        otp = generate_otp()
                        st.session_state.otp = otp
                        st.session_state.reset_email = email

                        try:
                          send_otp_email(email, otp)
                          st.success("✅ OTP sent successfully to your email.")
                          st.session_state.reset_mode = "otp"
                          st.rerun()
                        except Exception as e:
                          st.error(f"Failed to send OTP: {e}")
                      else:
                        st.error("❌ Email not found.")
            else:
                if st.session_state.get("reset_mode") == "sq":

                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")

                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")

                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Reset Password →", use_container_width=True):

                        if len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")

                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")

                        else:
                            with get_db() as c:
                                r = c.execute(
                                    "SELECT security_answer_hash FROM users WHERE email=?",
                                    (st.session_state.reset_email,)
                                ).fetchone()

                            if r and check_txt(ans, r[0]):
                                with get_db() as c:
                                    c.execute(
                                        "UPDATE users SET password_hash=? WHERE email=?",
                                        (hash_txt(npw), st.session_state.reset_email)
                                    )

                                st.success("✅ Password updated successfully!")
                                time.sleep(1)
                                st.session_state.reset_email = None
                                navigate("Login")

                            else:
                                st.error("❌ Incorrect security answer.")

                if st.session_state.get("reset_mode") == "otp":

                    st.info("📧 Enter the OTP sent to your registered email.")

                    entered_otp = st.text_input("Enter OTP")

                    npw = st.text_input(
                        "New password (min 8 chars)",
                        type="password",
                        key="otp_new"
                    )

                    confirm_npw = st.text_input(
                        "Confirm new password",
                        type="password",
                        key="otp_confirm"
                    )

                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Verify OTP & Reset Password", use_container_width=True):

                        if entered_otp != st.session_state.get("otp"):
                            st.error("❌ Invalid OTP.")

                        elif len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")

                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")

                        else:
                            with get_db() as c:
                                c.execute(
                                    "UPDATE users SET password_hash=? WHERE email=?",
                                    (hash_txt(npw), st.session_state.reset_email)
                                )

                            st.success("✅ Password updated successfully!")

                            st.session_state.otp = None
                            st.session_state.reset_email = None
                            st.session_state.reset_mode = None

                            time.sleep(1)
                            navigate("Login")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">🚢</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Intelligent Freight Quote Generation</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email=="infosys@ai" else "AI Freight Management"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">🚢 Intelligent Freight Quote Generation</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("### 👥 Registered Users")
        with get_db() as c:
          users = c.execute(
              "SELECT username, email FROM users ORDER BY username"
          ).fetchall()
        if users:
          st.table(users)
        else:
          st.info("No registered users found.")

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">🚢 Intelligent Freight Quote Generation</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "Quote Generation Accuracy", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [6]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# Get ngrok token
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# Pass Colab secrets to Streamlit
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")
os.environ["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")

# Stop old streamlit process
!pkill -f streamlit

# Wait a little
time.sleep(3)

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Reuse existing tunnel if it already exists
tunnels = ngrok.get_tunnels()

if tunnels:
    public_url = tunnels[0].public_url
    print("♻️ Reusing existing ngrok tunnel")
else:
    public_url = ngrok.connect(8501).public_url
    print("🚀 Created new ngrok tunnel")

print("=" * 60)
print("Intelligent Freight Quote Generation Live URL:", public_url)
print("=" * 60)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    process.terminate()
    !pkill -f streamlit
    ngrok.kill()

🚀 Created new ngrok tunnel
Intelligent Freight Quote Generation Live URL: https://freight-outback-papyrus.ngrok-free.dev
